# Issue Index Generator

Generate a JSON index file for all issues in a given provider's OCR collection.
This tool discovers all issues in a provider directory and creates a structured index
organized by media_alias → year → month → issue records.

The output index can be used as input file for the impresso text import pipeline.

## Import Required Libraries

In [1]:
import json
import os
import re
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple
import logging
from tqdm import tqdm
from datetime import date, datetime
import pandas as pd
from ast import literal_eval
from tqdm import tqdm
import ijson

from impresso_essentials.utils import ALL_MEDIA
# Register tqdm with pandas to enable progress_apply
tqdm.pandas()

## Configuration

Configure the provider collection parameters and output settings.
This changes for each provider's specific data situation

In [2]:
DATA_DIR = "../text_preparation/data"

## SUB case

In [5]:
# Provider name and media aliases mapping
PROVIDER_NAME = "SUB"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using SUB-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: ../text_preparation/data/issue_indices/issue_index.sub.json
Using SUB-specific directory structure discovery


### Load Provider Collection Data

Discover all OCR files in the provider directory structure.

In [6]:
dir_to_alias_sub = {
    "Hamburger_Echo": "hamb_echo",
    "Hamburger_Volksblatt": "hamb_volksbl",
    "Der_Hamburger_Beobachter": "hamb_beob",
    "Neue_hamburgische_Blaetter": "neu_hamb_bl",
    "Bergedorfer_Eisenbahn-Zeitung": "berg_eisb_z",
    "Norddeutsche_Zeitung": "norddt_ztg",
    "Hamburgischer_Correspondent": "hamb_corr",
    "Harburger_Anzeigen": "harb_anz",
    "Bergedorfer_Wochenblatt_zum_Nutzen_und_zur_Unterhaltung": "berg_wbl_nu",
    "Harburger_Anzeigen_und_Nachrichten": "harb_anz_nachr",
    "Altonaischer_Mercurius": "alt_merc",
    "Der_Hamburger_Beobachter_und_das_Archiv_Wissenschaften_und_Kuenste": "hamb_beob_arch",
    "Hamburgischer_Correspondent_und_Hamburgische_Boersen-Halle": "hamb_corr_boers",
    "Bergedorfer_Zeitung_und_Anzeiger": "berg_ztg_anz",
    "Eisenbahnzeitung": "eisb_ztg",
    "Relations-Courier": "rel_cour",
    "Hamburger_Tageblatt": "hamb_tagbl",
    "Stats-_und_gelehrte_Zeitung_des_hollsteinischen_unpartheyischen_Correspondenten": "stats_gel_holl",
    "Staats-_und_gelehrte_Zeitung_des_Hamburgischen_unpartheyischen_Correspondenten": "staats_gel_hamb",
    "Zuerst-bekandte_Schiffbecker_Stats-_und_gelehrte_Zeitung_des_hollsteinischen_unpartheyischen_Correspondenten": "zuerst_schiffb",
    "Gerichtszeitung": "gerichts_ztg",
    "Reichs-Post-Reuter": "reichs_post_r",
    "Billstedter_Zeitung": "billst_ztg",
    "Buergerzeitung": "buerger_ztg",
    "Bergedorfer_Zeitung": "berg_ztg",
    "Altonaer_Mercur": "alt_mercur",
    "Hamburger_Volkszeitung": "hamb_volksz",
    "Hamburger_Fremdenblatt": "hamb_fremdbl",
    "Privilegirter_Hollsteinischer_Avisen-Correspondente_durch_Europa_und_andere_Theile_der_Welt": "priv_holl_avis",
    "Hamburg-Altonaer_Volksblatt": "hamb_alt_volksbl",
    "Bergedorfer_Wochenblatt_und_Eisenbahn-Zeitung": "berg_wbl_eisb",
    "Hansische_Warte": "hans_warte",
    "Hamburger_Relations-Courier": "hamb_rel_cour",
    "Volksblatt_fuer_Harburg_Wilhelmsburg_und_Umgegend": "volksbl_harb",
    "Hamburgischer_Correspondent_und_neue_hamburgische_Boersen-Halle": "hamb_corr_neu"
}

In [5]:
any(v in ALL_MEDIA for v in dir_to_alias_sub.values() if v!='hamb_echo')

False

#### Define a sorting lambda function to assign edition letters 

In [6]:
# Edition sorting key: splits on '-', sorts by number first, then by edition type
# Returns (number, edition_priority) tuple
sort_edition = lambda e: (
    int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] else 999,
    {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1}.get(e.split('-')[-1] if '-' in e else e, 2)
)

# Test
test_editions = ["Ausgabe", "M1-Morgenausgabe", "A2-Abendausgabe", "M3-Morgenausgabe", "A1-Abendausgabe"]
print("Sorting with lambda:")
all_day_editions = sorted(test_editions, key=sort_edition)
for edition in test_editions:
    edition_l = chr(96 + all_day_editions.index(edition)+1)
    print(f"  {edition} → {sort_edition(edition)} --> {edition_l}")

Sorting with lambda:
  Ausgabe → (999, -1) --> e
  M1-Morgenausgabe → (1, 0) --> a
  A2-Abendausgabe → (2, 1) --> c
  M3-Morgenausgabe → (3, 0) --> d
  A1-Abendausgabe → (1, 1) --> b


### aggregate all in a function which creates and writes to disk the issue index

#### define a list of image format extensions

In [7]:
# sort the image formats from most desirable to least
possible_img_formats = [".jp2", ".tif", ".tiff", ".jpg", ".jpeg", ".pdf", ".png"]

In [ ]:
def get_issue_metadata_info(prov_base_dir, alias, issue_dir_path, base_dir: str = BASE_DATA_PATH, debug:bool=False):

    try:
        # extract the relative path to the issue
        parts = issue_dir_path.rstrip("/").split("/")
        year, month, day = parts[-4:-1]
        edition_name = parts[-1].split('/')[0]
            
        sort_editions = lambda e: (
            # there is a special case for Hamburger_Fremdenblatt/1942/08/19/: Abendausgabe  Auslandsausgabe  AV-Auslandsausgabe  Morgenausgabe  Spätausgabe
            int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] and e[1] != 'V' else 999,
            {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1, "Spätausgabe": 2, "Auslandsausgabe": 3}.get(e.split('-')[-1] if '-' in e else e, 2)
        )
        # For directories without explicit edition prefix
        # This will be resolved later when grouping by day
        day_dir = '/'.join(parts[:-1])
        all_day_editions = sorted([d for d in os.listdir(day_dir) if os.path.isdir(os.path.join(day_dir, d))], key=sort_editions)
        edition = chr(96 + all_day_editions.index(edition_name)+1)
        

        # IDENTIFY THE SUBDIR OF THE IMAGES
        non_xml_files = {'': [f for f in os.listdir(issue_dir_path) if not f.endswith('.xml')]}
        subdirs = [d for d in os.listdir(issue_dir_path) if os.path.isdir(os.path.join(prov_base_dir, d))]

        #if len(non_xml_files['']) == 0:
        # there are no other files than xmls check if there are subdirs which contain some
        for subdir in subdirs:
            non_xml_files[subdir] = [f for f in os.listdir(os.path.join(issue_dir_path, subdir)) if not f.endswith('.xml')]

        # IDENTIFY THE EXTENSION OF THE IMAGES FILES
        imgs_dir = []
        img_ext = []
        missing_image_files = None
        num_xml_files = len(list(Path(issue_dir_path).rglob('*.xml')))
        for subdir, files in non_xml_files.items():
            #if not imgs_dir and not img_ext:
                #print(f"ext: {ext}")
                #print(f"sum(ext in f for f in non_xml_files): {sum(ext in f for f in non_xml_files)+1}")
                #print(f"len(Path(issue_dir_path).rglob('*.xml'))+1: {len(list(Path(issue_dir_path).rglob('*.xml')))}")
            for ext in possible_img_formats:
                files_with_ext = [f for f in files if ext in f]
                if len(files_with_ext)+1 == num_xml_files:
                    # the extensions are sorted, the first one to match is the one of choice
                    imgs_dir.append(subdir)
                    img_ext.append(ext)
                    break
                elif debug:
                    print(f"WARNING!! subdir {issue_dir_path} and extension {ext} - found {len(files_with_ext)} image files, but there are {num_xml_files} xml files")
                if len(files_with_ext)!=0:
                    imgs_dir.append(subdir)
                    img_ext.append(ext)
                    # finding the exact page numbers of the images present
                    existing_image_files = [int(f.split('.')[0]) for f in files_with_ext if '.' in f and not f.startswith('.')]
                    missing_image_files = [i for i in range(1, num_xml_files) if i not in existing_image_files]
                    print(f"WARNING!! subdir {issue_dir_path} and extension {ext} - missing image files for pages {missing_image_files} - keeping note of it.")

        if len(imgs_dir)>1 or len(img_ext)>1:
            print(f"WARNING!! THERE MIGHT BE MULTIPLE IMAGE FORMATS; THE CHOSEN ONE WILL BE {img_ext[0]} PRESENT IN SUBDIR {imgs_dir[0]}, BUT THERE NEEDS TO BE AN XML CHECK!!")
        if len(imgs_dir)==0 or len(img_ext)==0:
            print(f"WARNING!! FOR PATH {issue_dir_path} WE ARE LIKELY MISSING ALL IMAGES; as img_ext={img_ext} and imgs_dir={imgs_dir}, THIS SHOULD BE CHECKED BY HAND, SETTING None.")
            imgs_dir.append(None)
            img_ext.append(None)
            

        # save the metadata for this issue
        issue_meta_info = {
            "day": day,
            "edition": edition,
            "local_path": [issue_dir_path.replace(base_dir, "")[1:]],
            "imgs_subdir": imgs_dir[0],
            "imgs_ext": img_ext[0]
        }

        if missing_image_files:
            issue_meta_info['missing_image_files'] = missing_image_files

        if debug:
            print(f"Adding issue {alias}-{year}-{month}-{day}-{edition} to the index: {issue_meta_info}")

        return year, month, issue_meta_info

    except Exception as e:
        print(f"Issue at path {issue_dir_path} presented an exception: {e}")
        raise e

In [15]:
def find_issues_w_glob(prov_base_dir, title_dirname, alias, base_dir: str = BASE_DATA_PATH, debug:bool=False) -> List[Dict]:
    """
    Discover SUB issues for one title using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]

    USING RGLOB
    
    Issues are identified by the presence of METS XML files containing "PPN".
    """
    alias_issues = {}

    all_mets_files = [str(p) for p in Path(os.path.join(prov_base_dir, title_dirname)).rglob("*/PPN*.xml")]
    print(f"Found {len(all_mets_files)} mets files for {alias}, sorting them by date...")
    # sort all issue files to write years and days in the correct order
    sorted_mets = sorted(all_mets_files, key=lambda x: date(int(x.rstrip("/").split("/")[-5]), int(x.rstrip("/").split("/")[-4]), int(x.rstrip("/").split("/")[-3])))

    print(f"Done sorting the mets files by date, now starting creating the index...")

    # look for all the mets files, which have a "PPN" in their filename -> this means the issue dir is also found
    for filepath in tqdm(sorted_mets):
        
        # extract the relative path to the issue
        issue_dir_path = os.path.dirname(filepath)

        year, month, issue_meta_info = get_issue_metadata_info(prov_base_dir, alias, issue_dir_path)

        # save the created metadata into the list of issues for this alias
        if year not in alias_issues:
            alias_issues[year] = {month: [issue_meta_info]}
        elif month not in alias_issues[year]:
            alias_issues[year][month] = [issue_meta_info]
        else:
            alias_issues[year][month].append(issue_meta_info)

    return alias_issues



def find_issues_w_walk(prov_base_dir, title_dirname, alias, base_dir: str = BASE_DATA_PATH, debug:bool=False) -> List[Dict]:
    """
    Discover SUB issues for one title using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]

    USING OS.WALK
    
    Issues are identified by the presence of METS XML files containing "PPN".
    """

    alias_issues = {}

    for issue_dir_path,dirs,files in tqdm(os.walk(os.path.join(prov_base_dir, title_dirname))):
        # only consider directories which contain files -> issue directories
        if files:

            if len([f for f in files if f.endswith('.xml')])==1:
                print(f"Warning! There is only one xml file in dir {issue_dir_path}, skipping this issue")
                continue

            year, month, issue_meta_info = get_issue_metadata_info(prov_base_dir, alias, issue_dir_path)
        
            # save the created metadata into the list of issues for this alias
            if year not in alias_issues:
                alias_issues[year] = {month: [issue_meta_info]}
            elif month not in alias_issues[year]:
                alias_issues[year][month] = [issue_meta_info]
            else:
                alias_issues[year][month].append(issue_meta_info)


    return alias_issues

In [16]:
def detect_issues_sub(out_path, base_dir: str = BASE_DATA_PATH, prov: str = PROVIDER_NAME, dir_to_alias_sub=dir_to_alias_sub, write_file:bool=True, debug:bool=False, resume=True, use_walk=True) -> List[Dict]:
    """
    Discover SUB newspaper issues using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]
    
    Issues are identified by the presence of METS XML files containing "PPN".
    
    Args:
        root_path: Root directory of the SUB collection
        
    Returns:
        List of issue metadata dictionaries with keys:
        {alias, year, month, day, edition, path, mets_file}
    """
    # First, list alias directories in base_dir for the provider
    prov_base_dir = os.path.join(base_dir, prov)

    try:
        title_dirs = [d for d in os.listdir(prov_base_dir) if os.path.isdir(os.path.join(prov_base_dir, d))]
    except OSError as e:
        print(f"Failed to list base directory {prov_base_dir}: {e}")
        return []

    if resume:
        with open(out_path, "r") as fin:
            all_issues = json.load(fin)
            print(f"The following aliases were already processed and will be skipped: {all_issues.keys()}")
    else:
        all_issues = {}

    for title_dirname, alias in dir_to_alias_sub.items():

        if alias in all_issues:
            print(f"Alias {alias} is already present in the index and will be skipped.")
            continue
        #elif alias == "hamb_corr":
        #    print(f"Skipping hamb_corr for now as it was taking too much time.")
        #    continue
        else:
            print(f"Processing to find the issues of alias {alias}...")

        if use_walk:    
            alias_issues = find_issues_w_walk(prov_base_dir, title_dirname, alias, base_dir, debug)
        else:
            alias_issues = find_issues_w_glob(prov_base_dir, title_dirname, alias, base_dir, debug)


        print(f"Adding {sum(len(d) for m in alias_issues.values() for d in m.values())} issues for alias {alias} to the total list of aliases")
        all_issues[alias] = alias_issues

        if write_file:
            with open(out_path, "w") as fout:
                json.dump(all_issues, fout, indent=4)

    return all_issues

In [ ]:
all_sub_issues = detect_issues_sub(OUTPUT_FILE)

Now sort all the results by year, month, day and edition letter to ease the use of the index.

In [7]:
with open(OUTPUT_FILE, "r") as fin:
    all_issues = json.load(fin)

In [12]:
all_sorted_issues = {}
duplicated_issues = {}
for alias, year_dicts in tqdm(all_issues.items()):
    all_sorted_issues[alias] = {}
    sorted_years = sorted(year_dicts.keys(), key=lambda x: int(x))
    for year in sorted_years:
    #for year, month_dicts in year_dicts.items():
        all_sorted_issues[alias][year] = {}
        sorted_months = sorted(year_dicts[year], key=lambda x: int(x))
        for month in sorted_months:
        #for month, days in month_dicts.items():
            sorted_issues = sorted(year_dicts[year][month], key=lambda x: int(x["day"]))
            if any(i['edition']==j['edition'] for i in sorted_issues for j in sorted_issues if i!=j and i['day']==j['day']):
                print(f"Warning! duplicated editions in the issues of {alias}-{year}-{month}: {sorted_issues}")
                duplicated_issues[{alias}] = [year, month]

            # now save the newly sorted issues
            all_sorted_issues[alias][year][month] = sorted_issues

  0%|          | 0/35 [00:00<?, ?it/s]

100%|██████████| 35/35 [00:01<00:00, 22.62it/s]


In [14]:

with open(OUTPUT_FILE, "w") as fout:
    json.dump(all_sorted_issues, fout, indent=4)

## RTS Case

In [2]:


# Provider name and media aliases mapping
PROVIDER_NAME = "RTS"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using SUB-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: ../../../data/issue_indices/issue_index.rts.json
Using SUB-specific directory structure discovery


### Read the processed metadata file, and filter out shows without audio files 

(also write this filtered version to disk)

In [ ]:
metadata_prep_filename = f"{DATA_DIR}/sample_data/RTS/unique_programs_metadata.rts.csv"

rts_metadata_df = pd.read_csv(metadata_prep_filename)
print(len(rts_metadata_df))
rts_metadata_df.head()

First remove the unused columns in this context.

The issue index is only used as a pointer to the data for each issue. 
Another dict will be created to store the metadata necessary for the canonical ingestion

In [ ]:
index_cols = ['alias', 'date_str', 'stripped_OID', 'stt_filename', 'final_mp3_filename']
df_for_index = rts_metadata_df[index_cols].copy()

df_for_index.head()

### Find duplicated values

In [17]:
duplicated = df_for_index[df_for_index.duplicated()]
print(f"There are {len(duplicated)} duplicated values in this data!, in the aliases {set(duplicated.alias.values)}")

There are 64 duplicated values in this data!, in the aliases {'ombres_eco', 'mag_eco'}


In [23]:
no_dupli = df_for_index.drop_duplicates()

# sort them by date
no_dupli['date_dt'] = pd.to_datetime(no_dupli['date_str'], format="%d/%m/%Y")
no_dupli = no_dupli.sort_values(by=['alias', 'date_dt'])

print(f"There are {len(no_dupli)} non-duplicated records.")

There are 26161 non-duplicated records.


In [ ]:
no_dupli

### Create the issue index file

In [ ]:
index_dict = no_dupli.to_dict('index')
index_dict

In [26]:
def get_edition(ed_num):
    if ed_num <= 26: 
        return chr(96+ed_num)
    # only works for editions > 26
    second_letter = ed_num%26
    first_letter = int(ed_num/26)
    if second_letter == 0:
        first_letter -= 1
        second_letter = 26
    #print(f"first_letter={first_letter}, second_letter={second_letter}")
    return chr(96+first_letter)+chr(96+second_letter)

In [ ]:
# check that it works
for i in range(25, 100):
    print(f"num={i}, ed:{get_edition(i)}")

In [ ]:
#rts_base_path = os.path.join(BASE_DATA_PATH, PROVIDER_NAME)
all_rts_issues = {alias: {} for alias in df_for_index['alias'].drop_duplicates().values}
# initialize the same way
num_eds_per_day = {alias: {} for alias in df_for_index['alias'].drop_duplicates().values}

for idx, row_dict in index_dict.items():

    day, month, year = row_dict['date_str'].split('/')

    alias = row_dict['alias']

    ## defining the edition
    if row_dict['date_str'] in num_eds_per_day[alias]:
        # there are already some issues for that day, increase by one the value and deduct the correct edition
        num_eds_per_day[alias][row_dict['date_str']] += 1
        # if there are more than 26 editions for a given day, the edition lettes should combine (aa, ab,...)
        issue_ed = get_edition(num_eds_per_day[alias][row_dict['date_str']])
    else:
        # if no issue already exists for that alias on that day, init the value 
        num_eds_per_day[alias][row_dict['date_str']] = 1
        issue_ed = 'a' 

    issue_dict = {
        "day": day,
        "edition": issue_ed,
        # the local path is the one of the MP3 file 
        "local_path": [os.path.join(PROVIDER_NAME, alias, "audio", row_dict['final_mp3_filename'])],
        "ext": ".mp3",
        "stripped_OID": row_dict['stripped_OID'],
        "xml_filepath": os.path.join(PROVIDER_NAME, alias, "stt", row_dict['stt_filename']),
    }

    # set the issue dict in the overall dict
    if year in all_rts_issues[alias]:
        if month in all_rts_issues[alias][year]:
            all_rts_issues[alias][year][month].append(issue_dict)
        else:
            all_rts_issues[alias][year][month] = [issue_dict]
    else:
        all_rts_issues[alias][year] = {month: [issue_dict]}


all_rts_issues

Save the resulting dict to disk

In [32]:
with open(OUTPUT_FILE, "w") as fout:
    json.dump(all_rts_issues, fout, indent=4)

#### Check the cases where there is no valid file

for each of these files, we will try to find if there is any file which matches the OID

In [ ]:
last_alias = None
for row in no_existing_mp3s.itertuples():
    print(row.alias, row.stripped_OID, row.spt_filenames, row.mp3_filenames)
    if row.alias!=last_alias:
        # update the values if we changed alias
        audios_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "audio")
        stt_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "stt")
        last_alias = row.alias
        print(f"Changed values of audios_dir to {audios_dir} and stt_dir to {stt_dir}. Listing their contents")
        all_audios = os.listdir(audios_dir)
        all_stt = os.listdir(stt_dir)

    #matching_stt = row.stripped_OID
    stt_file = f"{row.stripped_OID}_STT.xml"
    print(f"{row.alias}: {stt_file in all_stt} - stt_file={stt_file}")

    support_filenames = [f for f in literal_eval(row.spt_filenames) if f]
    all_valid_audios = []
    for f in support_filenames:
        if f and '.wav' in f:
            print(f"removing the .wav in {f}")
            f = f.replace(".wav", '')
        valid_audios = [audio for audio in all_audios if f in audio]
        print(f"filename {f} corresponds to audio {valid_audios}!")
        all_valid_audios.extend(valid_audios)
    
    oid_also_in = [audio for audio in all_valid_audios if row.stripped_OID.lower() in audio]
    if len(all_valid_audios)>1:
        print(f"{row.alias}: There are more than 1 audio file which matches: {all_valid_audios}")
    elif len(all_valid_audios)==1:
        print(f"{row.alias}: audio_file={all_valid_audios[0]}")
    else:
        print(f"{row.alias}: No valid audio file")

## INA case

In [21]:
# Provider name and media aliases mapping
PROVIDER_NAME = "INA"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original/{PROVIDER_NAME}/Data"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}.json"


print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")

Provider path: /mnt/project_impresso/original/INA/Data
Output file: ../text_preparation/data/issue_indices/issue_index.ina.json


### Read in the preprocessed file

In [ ]:
all_issues_df_path = f"{DATA_DIR}/sample_data/INA/all_issues_v2.csv"
df_for_index = pd.read_csv(all_issues_df_path, index_col=0)
df_for_index

#### Generating the issue index

In [16]:
# create a list of dicts view for the issues
index_dict = df_for_index.to_dict('index')
# obtain the list of aliases
ina_aliases = list(df_for_index['issue_id'].apply(lambda x: x.split('-')[0]).drop_duplicates().values)
len(ina_aliases)

44

In [ ]:
all_ina_issues = {alias: {} for alias in ina_aliases}

for idx, row_dict in index_dict.items():

    # the issue edition disambiguation was already done in the metadata preprocessing.
    alias, year, month, day, edition = row_dict['issue_id'].split('-')

    issue_dict = {
        "day": day,
        "edition": edition,
        # the local path is the one of the MP3 file 
        "local_path": literal_eval(row_dict['local_path']),
        "ext":  row_dict['extension'],
        "notice_id": row_dict['notice'],
        "xml_filepath": literal_eval(row_dict['xml_filepath']),
    }

    # there is one issue which had the wrong filename in the provided metadata
    if row_dict["issue_id"] == "TrParis-1953-12-10-a":
        issue_dict["local_path"] = [
            f.replace("_02_","_04_") for f in issue_dict["local_path"]
        ]
        issue_dict["xml_filepath"] = [
            f.replace("_02_","_04_") for f in issue_dict["xml_filepath"]
        ]

    # set the issue dict in the overall dict
    if year in all_ina_issues[alias]:
        if month in all_ina_issues[alias][year]:
            all_ina_issues[alias][year][month].append(issue_dict)
        else:
            all_ina_issues[alias][year][month] = [issue_dict]
    else:
        all_ina_issues[alias][year] = {month: [issue_dict]}


all_ina_issues

In [28]:
with open(OUTPUT_FILE, "w") as fout:
    json.dump(all_ina_issues, fout, indent=4)

## KBR case

In [ ]:
# Provider name and media aliases mapping
PROVIDER_NAME = "KBR"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original/{PROVIDER_NAME}"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/{PROVIDER_NAME}/issues_index.json"

# Pattern for extracting metadata from filenames
# Adjust these patterns based on your provider's naming convention
# Example pattern: 1924587_newspaper_actionfem_1927-10-15_01
FILENAME_PATTERN = r'.*?_newspaper_([a-z]+)_(\d{4})-(\d{2})-(\d{2})_(\d{2})'

# Example: For Olive format with dates in directory names
# FILENAME_PATTERN = r'(\d{4})/(\d{2})/(\d{2})/.*'

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Pattern: {FILENAME_PATTERN}")

## BNF case

In [7]:
# Provider name and media aliases mapping
PROVIDER_NAME = "BNF"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"{DATA_DIR}/issue_indices/issue_index.{PROVIDER_NAME.lower()}_new.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using BNF-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: ../text_preparation/data/issue_indices/issue_index.bnf_new.json
Using BNF-specific directory structure discovery


In [4]:
def find_bnf_issues_w_walk(prov_base_dir, alias, base_dir: str = BASE_DATA_PATH, skipped_issues:list|None=None) -> List[Dict]:
    """
    Discover BNF issues for one title using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_letter]

    USING OS.WALK
    
    Issues are identified by the presence of METS XML files containing "PPN".
    """

    alias_issues = {}
    alias_dir = os.path.join(prov_base_dir, alias)
    for issue_dir_path,dirs,files in tqdm(os.walk(alias_dir)):
        # only consider directories which contain files -> issue directories
        if alias=='oeuvre' and ('toc' in issue_dir_path or 'ocr' in issue_dir_path or (dirs and files)):
            # if we are working with oeuvre, some issue_dirs have the "old" format and should be skipped
            # this can be identified because the old format had a dir per issue which contains other directories
            skipped_issues.append(issue_dir_path)
            continue
        elif files:
            manifest_file = [f for f in files if f=='manifest.json']
            if len(manifest_file)==0:
                print(f"Warning! Manifest JSON file is missing in dir {issue_dir_path}, skipping this issue (dirs={dirs}, files={files})")
                continue
            elif len(manifest_file)>1:
                print(f"Warning! More than 1 Manifest JSON file in dir {issue_dir_path}, using the first one. (dirs={dirs}, files={files})")

            # the issue edition disambiguation was already done in the download
            #print(f"issue_dir_path.replace(alias_dir, ''): {issue_dir_path.replace(alias_dir, '')}")
            year, month, day, edition = issue_dir_path.replace(alias_dir, '')[1:].split('/')

            manifest_path = os.path.join(issue_dir_path, manifest_file[0])
            with open(manifest_path, 'rb') as f:
                id_url = next(ijson.items(f, 'id'))
        
            issue_dict = {
                "day": day,
                "edition": edition,
                # the local path is the one of the MP3 file 
                "local_path": [issue_dir_path.replace(base_dir, '')[1:]],
                "ark_id": id_url.split('/')[-2],
                "batch": "BNF_API_NEW"
            }
        
            # save the created metadata into the list of issues for this alias
            if year not in alias_issues:
                alias_issues[year] = {month: [issue_dict]}
            elif month not in alias_issues[year]:
                alias_issues[year][month] = [issue_dict]
            else:
                alias_issues[year][month].append(issue_dict)


    return alias_issues, skipped_issues

In [12]:
def detect_issues_bnf(out_path, base_dir: str = BASE_DATA_PATH, prov: str = PROVIDER_NAME, write_file:bool=True, debug:bool=False, resume=True, use_walk=True, aliases_to_override=None) -> List[Dict]:
    """
    Discover BNF newspaper issues using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]
    
    Issues are identified by the presence of METS XML files containing "PPN".
    
    Args:
        root_path: Root directory of the SUB collection
        
    Returns:
        List of issue metadata dictionaries with keys:
        {alias, year, month, day, edition, path, mets_file}
    """
    # the titles which were already shared with us for Impresso 1 have another internal organisation
    imp1_dirname_to_alias = {
        'Excelsior': 'excelsior',
        'La-Fronde': 'lafronde',
        'Marie-Claire': 'marieclaire',
        'Oeuvre': 'oeuvre', # oeuvre contains both old and new data - both need to be listed in issue_index
    }

    # First, list alias directories in base_dir for the provider
    prov_base_dir = os.path.join(base_dir, prov)

    try:
        title_dirs = [d for d in os.listdir(prov_base_dir) if os.path.isdir(os.path.join(prov_base_dir, d)) and '2020' not in d and not d.startswith('.')]
    except OSError as e:
        print(f"Failed to list base directory {prov_base_dir}: {e}")
        return []

    print(f"Found {len(title_dirs)} title directories: {title_dirs}")

    if resume and os.path.exists(out_path):
        with open(out_path, "r") as fin:
            all_issues = json.load(fin)
            print(f"The following aliases were already processed and will be skipped: {all_issues.keys()}")
    else:
        all_issues = {}

    oeuvre_skipped_issues = None
    if aliases_to_override is None:
        aliases_to_override = []

    for idx, alias in enumerate(title_dirs):

        if alias in imp1_dirname_to_alias:
            # replace with the correct alias
            alias = imp1_dirname_to_alias[alias]
        
        if alias in all_issues and alias not in aliases_to_override:
            print(f"{idx+1}/{len(title_dirs)} - Alias {alias} is already present in the index and will be skipped.")
            continue
        else:
            print(f"{idx+1}/{len(title_dirs)} - Processing to find the issues of alias {alias}...")

        if alias not in ['excelsior', 'lafronde', 'marieclaire']:    
            # skip the impresso 1 data for now, will add later
            if alias == 'oeuvre': 
                alias_issues, oeuvre_skipped_issues = find_bnf_issues_w_walk(prov_base_dir, alias, base_dir, [])
            else:
                alias_issues, _ = find_bnf_issues_w_walk(prov_base_dir, alias, base_dir, None)

        print(f"Adding {sum(len(d) for m in alias_issues.values() for d in m.values())} issues for alias {alias} to the total list of aliases")
        all_issues[alias] = alias_issues

        if write_file:
            with open(out_path, "w") as fout:
                json.dump(all_issues, fout, indent=4)

    return all_issues, oeuvre_skipped_issues

In [8]:
OUTPUT_FILE

'../text_preparation/data/issue_indices/issue_index.bnf_new.json'

In [9]:
aliases_to_override = ["revuecolan", "rhcf", "togocameroun"]

In [13]:
all_sub_issues, skipped_issues = detect_issues_bnf(OUTPUT_FILE, aliases_to_override=aliases_to_override)

Found 129 title directories: ['jeuneeurope1930', 'letemps', 'quinzainecol', 'vsve', 'petitepresse', 'petitmarocain', 'bsgecm', 'lescolonies', 'pressecolill', 'lafrique1844', 'liberation', 'lecaucase', 'unionfrancaise', 'jeuneeuroperev', 'combat', 'franceyougoslavie', 'ikdam', 'democratiepacifique', 'terreeurope', 'europeor1919', 'gazettecoloniale', 'bocfo', 'nyherald', 'cridespeuples', 'etatsuniseurope', 'europedabord', 'europecoloniale', 'lepays', 'libertecol', 'lesbalkans', 'parisbalkans', 'demokratischeztg', 'depechetoulouse', 'progrescol', 'bulletinoffcol', 'revuecolan', 'Oeuvre', 'europedanubienne', 'laliberte', 'figarosupl', 'armeecoloniale', 'lacharente', 'elouma', 'chicagotribune', 'europeindcom', 'europefuture', 'nouvellestcheco', 'parismidi', 'europeartistique', 'lepeuple', 'notretemps', 'europefinanciere1', 'candide', 'francerussie', 'courrierlondres', 'bulletincol', 'mondecolill', 'lacroix', 'intransigeant', 'lecourrier', 'lapresse', 'vendredi', 'europenouvelle', 'europefin

91it [00:00, 133.83it/s]


Adding 27 issues for alias revuecolan to the total list of aliases
37/129 - Alias oeuvre is already present in the index and will be skipped.
38/129 - Alias europedanubienne is already present in the index and will be skipped.
39/129 - Alias laliberte is already present in the index and will be skipped.
40/129 - Alias figarosupl is already present in the index and will be skipped.
41/129 - Alias armeecoloniale is already present in the index and will be skipped.
42/129 - Alias lacharente is already present in the index and will be skipped.
43/129 - Alias elouma is already present in the index and will be skipped.
44/129 - Alias chicagotribune is already present in the index and will be skipped.
45/129 - Alias europeindcom is already present in the index and will be skipped.
46/129 - Alias europefuture is already present in the index and will be skipped.
47/129 - Alias nouvellestcheco is already present in the index and will be skipped.
48/129 - Alias parismidi is already present in the

87it [00:00, 372.67it/s]


Adding 31 issues for alias togocameroun to the total list of aliases
74/129 - Alias leconstitutionnel is already present in the index and will be skipped.
75/129 - Alias vielatine is already present in the index and will be skipped.
76/129 - Alias nouvellefrancemars is already present in the index and will be skipped.
77/129 - Alias lepetitjournal is already present in the index and will be skipped.
78/129 - Alias renaissancecol is already present in the index and will be skipped.
79/129 - Alias actionfrancaise1908 is already present in the index and will be skipped.
80/129 - Alias rqcm is already present in the index and will be skipped.
81/129 - Alias annalescol is already present in the index and will be skipped.
82/129 - Alias europeorroum is already present in the index and will be skipped.
83/129 - Alias grandechonord is already present in the index and will be skipped.
84/129 - Alias europeillrev is already present in the index and will be skipped.
85/129 - Alias bouffon is alre

132it [00:00, 160.75it/s]


Adding 52 issues for alias rhcf to the total list of aliases
100/129 - Alias defensenationalparis is already present in the index and will be skipped.
101/129 - Alias univers is already present in the index and will be skipped.
102/129 - Alias abendland is already present in the index and will be skipped.
103/129 - Alias canardenchaine is already present in the index and will be skipped.
104/129 - Alias courriermarna is already present in the index and will be skipped.
105/129 - Alias humanite is already present in the index and will be skipped.
106/129 - Alias tablettescol is already present in the index and will be skipped.
107/129 - Alias europejq is already present in the index and will be skipped.
108/129 - Alias cripeuple1871 is already present in the index and will be skipped.
109/129 - Alias europepscil is already present in the index and will be skipped.
110/129 - Alias echangouleme is already present in the index and will be skipped.
111/129 - Alias paixtravail is already pre